# 07 · The library on this machine: eleven families, four sets, three rungs

Every model family below is tuned on every predictor set at every rung, by the same procedure:

- **Optuna's TPE sampler**, seed 2026, a fixed budget of trials per family (printed below);
- each trial is scored by **survey-weighted average precision, out of fold**, over the five
  development folds of notebook 03, with survey weights rescaled to average one in each fold;
- the best settings are refitted on all 5,518 development women and the held-out women are
  scored once.

Then, for every tuned model:

- two **recalibrated** versions, **isotonic** (a flexible step map) and **Platt** (a logistic map,
  which keeps the ranking), each fitted inside the development folds so it is never scored on the
  women it was fitted to;
- a **stacked ensemble** (a logistic regression over the five best models' out-of-fold
  predictions) and a **cumulative-link ensemble** (the best model refitted at contact thresholds
  1 to 10, forced to fall as the threshold rises, pooled and mapped back to a chance).

The runner (`scripts/train_library.py`) works through the jobs in order of what each needs,
saves every model as it finishes, and skips what is done. **This takes days**; stop and re-run
the cell at will. CatBoost and the networks need a GPU and are left to notebook 08.

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

WORKERS = 4   # models trained at once; each uses up to 5 cores for its folds

In [ ]:
print(pd.DataFrame([{"family": f, "trials": tuning.TRIALS[f]} for f in lib.FAMILIES]).to_string(index=False))
print(f"\n{len(lib.ALL_SETS)} predictor sets x {len(lib.RUNGS)} rungs")
repro.run_script("train_library.py", "--root", P.results, "--status")

In [ ]:
repro.run_script("train_library.py", "--root", P.results, "--device", "cpu", "--workers", WORKERS)

In [ ]:
repro.run_script("train_library.py", "--root", P.results, "--status")